# Set Cover: Gate vs. Readout vs. Gate & Readout Errors

This notebook compares the effect of having gate errors, readout errors, and both across **12 distinct 9-qubit Set Cover instances**.

The ansatz construction uses the **momentum_sa_merged** approach.

Each type of error is run for **5 trials per Hamiltonian**. Results are aggregated (mean ± std) and compared per instance.

In [ ]:
%pip install -U qiskit_ibm_runtime

In [1]:
import sys
import os
import threading
import time
import math
import heapq
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator

from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit.primitives import BackendEstimatorV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# --- Path Setup ---
vqa_root = os.path.abspath(os.path.join('../../../../'))
if vqa_root not in sys.path:
    sys.path.insert(0, vqa_root)

ansatz_pruning_dir = os.path.abspath(os.path.join('../../../'))
if ansatz_pruning_dir not in sys.path:
    sys.path.insert(0, ansatz_pruning_dir)

from AnsatzPruning import MomentumMonteCarlo
from AnsatzPruning.MomentumBuilder import momen_layer
from Optimization import MonteCarlo

print('Imports OK')

Imports OK


## 1. Set Cover Hamiltonians

Twelve distinct 9-qubit Set Cover instances (9 subsets = 9 qubits each).  
All instances are constructed so that at least one valid cover exists (ground state energy = 0).

*The benchmarking instances are currently the same as in **TestSetCoverMerged.ipynb**.

In [2]:
def get_subset_Hamiltonian(universe, subsets):
    num_subsets = len(subsets)
    total_op = SparsePauliOp(["I" * num_subsets], coeffs=[0.0])
    for element in universe:
        relevant_indices = [i for i, s in enumerate(subsets) if element in s]
        pauli_id = "I" * num_subsets
        total_op += SparsePauliOp([pauli_id], coeffs=[1.0])
        for idx in relevant_indices:
            z_str = ["I"] * num_subsets
            z_str[num_subsets - 1 - idx] = "Z"
            total_op += SparsePauliOp(["".join(z_str)], coeffs=[0.5])
            total_op += SparsePauliOp([pauli_id], coeffs=[-0.5])
        relevant_indices.sort()
        for i_idx in range(len(relevant_indices)):
            for j_idx in range(i_idx + 1, len(relevant_indices)):
                u, v = relevant_indices[i_idx], relevant_indices[j_idx]
                total_op += SparsePauliOp([pauli_id], coeffs=[0.5])
                z_u = ["I"] * num_subsets; z_u[num_subsets - 1 - u] = "Z"
                total_op += SparsePauliOp(["".join(z_u)], coeffs=[-0.5])
                z_v = ["I"] * num_subsets; z_v[num_subsets - 1 - v] = "Z"
                total_op += SparsePauliOp(["".join(z_v)], coeffs=[-0.5])
                z_uv = ["I"] * num_subsets
                z_uv[num_subsets - 1 - u] = "Z"; z_uv[num_subsets - 1 - v] = "Z"
                total_op += SparsePauliOp(["".join(z_uv)], coeffs=[0.5])
    return total_op.simplify()


# --- 12 distinct 9-qubit instances ---
# All are verified to have at least one valid cover (ground state energy = 0)
INSTANCES = {
    # ---- universe size 6 ----
    "A: pairs + triples (u=6)": {
        "universe": [1, 2, 3, 4, 5, 6],
        "subsets": [{1,2}, {3,4}, {5,6}, {1,3,5}, {2,4,6}, {1,4}, {2,5}, {3,6}, {1,2,6}],
    },
    "B: chain + cross (u=6)": {
        "universe": [1, 2, 3, 4, 5, 6],
        "subsets": [{1,3}, {2,4}, {5,6}, {1,2,5}, {3,4,6}, {2,3}, {1,6}, {4,5}, {2,6}],
    },
    "C: halves + singles (u=6)": {
        "universe": [1, 2, 3, 4, 5, 6],
        "subsets": [{1,2,3}, {4,5,6}, {1,4}, {2,5}, {3,6}, {1,5}, {2,6}, {3,4}, {1,2,6}],
    },
    # ---- universe size 5 ----
    "D: chain (u=5)": {
        "universe": [1, 2, 3, 4, 5],
        "subsets": [{1,2}, {2,3}, {3,4}, {4,5}, {1,3,5}, {2,4}, {1,4}, {3,5}, {1,2,4}],
    },
    "E: random overlap (u=5)": {
        "universe": [1, 2, 3, 4, 5],
        "subsets": [{1,3}, {2,4}, {3,5}, {1,5}, {2,3}, {1,2,4}, {3,4,5}, {1,2,5}, {2,4,5}],
    },
    # ---- universe size 4 ----
    "F: dense pairs/triples (u=4)": {
        "universe": [1, 2, 3, 4],
        "subsets": [{1,2}, {1,3}, {1,4}, {2,3}, {2,4}, {3,4}, {1,2,3}, {2,3,4}, {1,2,4}],
    },
    # ---- universe size 7 ----
    "G: even split (u=7)": {
        "universe": [1, 2, 3, 4, 5, 6, 7],
        "subsets": [{1,2,3}, {4,5,6,7}, {1,4,7}, {2,5}, {3,6}, {1,5,7}, {2,4,6}, {3,5}, {1,2,7}],
    },
    "H: cross-pairs (u=7)": {
        "universe": [1, 2, 3, 4, 5, 6, 7],
        "subsets": [{1,2,4}, {3,5,7}, {2,6}, {1,4,6}, {3,5}, {2,7}, {1,3,6}, {4,5,7}, {1,2,6}],
    },
    # ---- universe size 8 ----
    "I: quad split (u=8)": {
        "universe": [1, 2, 3, 4, 5, 6, 7, 8],
        "subsets": [{1,2}, {3,4}, {5,6}, {7,8}, {1,3,5,7}, {2,4,6,8}, {1,4,6}, {2,3,7}, {5,8}],
    },
    "J: stripes (u=8)": {
        "universe": [1, 2, 3, 4, 5, 6, 7, 8],
        "subsets": [{1,3,5,7}, {2,4,6,8}, {1,2}, {3,4}, {5,6}, {7,8}, {1,4,7}, {2,5,8}, {3,6}],
    },
    # ---- universe size 9 ----
    "K: 3x3 grid (u=9)": {
        "universe": [1, 2, 3, 4, 5, 6, 7, 8, 9],
        "subsets": [{1,2,3}, {4,5,6}, {7,8,9}, {1,4,7}, {2,5,8}, {3,6,9}, {1,5,9}, {2,4,8}, {3,6,7}],
    },
    "L: singletons + big sets (u=6)": {
        "universe": [1, 2, 3, 4, 5, 6],
        "subsets": [{1}, {2}, {3}, {4}, {5}, {6}, {1,2,3}, {4,5,6}, {1,4}],
    },
}

# Build Hamiltonians and print ground-state info
hamiltonians = {}
ground_energies = {}

for name, cfg in INSTANCES.items():
    H = get_subset_Hamiltonian(cfg["universe"], cfg["subsets"])
    nq = len(cfg["subsets"])
    diag = np.real(np.diag(H.to_matrix()))
    e_min = np.min(diag)
    solutions = np.where(np.isclose(diag, e_min))[0]
    hamiltonians[name] = H
    ground_energies[name] = e_min
    print(f"{name}: {nq} qubits | ground energy = {e_min:.4f} | {len(solutions)} solution(s)")

A: pairs + triples (u=6): 9 qubits | ground energy = 0.0000 | 3 solution(s)
B: chain + cross (u=6): 9 qubits | ground energy = 0.0000 | 4 solution(s)
C: halves + singles (u=6): 9 qubits | ground energy = 0.0000 | 3 solution(s)
D: chain (u=5): 9 qubits | ground energy = 0.0000 | 2 solution(s)
E: random overlap (u=5): 9 qubits | ground energy = 0.0000 | 2 solution(s)
F: dense pairs/triples (u=4): 9 qubits | ground energy = 0.0000 | 3 solution(s)
G: even split (u=7): 9 qubits | ground energy = 0.0000 | 2 solution(s)
H: cross-pairs (u=7): 9 qubits | ground energy = 0.0000 | 1 solution(s)
I: quad split (u=8): 9 qubits | ground energy = 0.0000 | 3 solution(s)
J: stripes (u=8): 9 qubits | ground energy = 0.0000 | 3 solution(s)
K: 3x3 grid (u=9): 9 qubits | ground energy = 0.0000 | 3 solution(s)
L: singletons + big sets (u=6): 9 qubits | ground energy = 0.0000 | 5 solution(s)


# 2. Noise Model Setup

In [3]:
fake_torino = FakeTorino()

simulators = {
    "combined": AerSimulator.from_backend(fake_torino),
    "gate": AerSimulator.from_backend(fake_torino, gate_error=True, readout_error=False),
    "readout": AerSimulator.from_backend(fake_torino, gate_error=False, readout_error=True),
    "ideal": AerSimulator.from_backend(fake_torino, gate_error=False, readout_error=False)
}

estimators = {
    name: BackendEstimatorV2(backend=simulator)
    for name, simulator in simulators.items()
}

pass_managers = {
    name: generate_preset_pass_manager(optimization_level=1, backend=fake_torino, seed_transpiler=123)
    for name, simulator in simulators.items()
}

backend_configs = {
    name: {
        "estimator": estimators[name],
        "pass_manager": pass_managers[name],
    }
    for name in estimators
}

print(f"Active noise loops prepared for evaluation: {list(backend_configs.keys())}")

Active noise loops prepared for evaluation: ['combined', 'gate', 'readout', 'ideal']


In [4]:
def cost_func_local(params, circuit, hamiltonian, backend_config):
    isa_circuit = backend_config["pass_manager"].run(circuit)
    isa_observable = hamiltonian.apply_layout(isa_circuit.layout)

    estimator = backend_config["estimator"]

    return estimator.run([(isa_circuit, isa_observable, params)]).result()[0].data.evs


def gradi_local(i, params, circuit, hamiltonian, backend_config):
    delta = np.zeros(len(params))
    delta[i] = math.pi / 2
    costp = cost_func_local(params+delta, circuit, hamiltonian, backend_config)
    costm = cost_func_local(params-delta, circuit, hamiltonian, backend_config)
    return (costp - costm) / 2


def momentum_sa_merged_noisy(params:list, inds:list, ansatz:QuantumCircuit,
                        circuit:QuantumCircuit, hamiltonian:SparsePauliOp,
                        backend_config:dict, beta1:float, beta2:float,
                        iters:int=2, optimization_runs:int=100):
    num_qubits = circuit.num_qubits
    observables = [*hamiltonian.paulis, hamiltonian]
    M = np.zeros((len(params))) # Momentum
    currCirc = QuantumCircuit(num_qubits)
    currCirc = currCirc.compose(ansatz)

    for iter in range(iters):
        # Calculate momentum
        accumulator = []
        for i in range(len(params)):
            grad_i = abs(gradi_local(i, params, currCirc, hamiltonian, backend_config)).item()
            M[i] = beta1 * M[i] + (1-beta1) * grad_i
            heapq.heappush(accumulator, (-M[i], inds[i]))

        # Construct momentum layer and append it to circuit
        keep = max(2, num_qubits // 2) # keep = how many qubits with highest momentums we use
        momentum_layer, new_params, new_inds = momen_layer(iter, num_qubits, accumulator, keep=keep)
        params = params + new_params
        inds = inds + new_inds
        M = np.concatenate((M, len(new_params)*[0]))
        ansatz = ansatz.compose(momentum_layer)
        currCirc = circuit.compose(ansatz)

        # Run simulated annealing to optimize params
        simulator = backend_config["estimator"].backend
        sa_params = MonteCarlo.simulated_annealing(
            optimization_runs, np.array(params), currCirc, simulator, observables, backend_config["estimator"]
        )
        params = list(sa_params)

    circuit = circuit.compose(ansatz)
    cost_final = cost_func_local(params, circuit, observables, backend_config)
    energy_final = cost_final[-1] # Last element in the list is the energy
    
    return energy_final

# 3. Helpers

In [5]:
def make_initial_ansatz(num_qubits):
    params_symbols = [Parameter(f'a{i}') for i in range(num_qubits)]
    circuit = QuantumCircuit(num_qubits)
    ansatz = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        ansatz.rx(params_symbols[i], i)
    return circuit, ansatz, [1.0] * num_qubits, list(range(num_qubits))

def execute_concurrent_benchmarks(task_dicts):
    threads = []
    outputs = {}

    # Internal worker function to time each thread
    def thread_worker(task_name, func_header, args):
        start_time = time.perf_counter()
        output = func_header(*args)
        end_time = time.perf_counter()
        outputs[task_name] = {
            "output": output,
            "runtime": end_time - start_time
        }

    for task in task_dicts:
        thread = threading.Thread(
            target = thread_worker,
            args=(task["name"], task["func"], task["args"])
        )
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    return outputs

# 4. Run 5 Trials in 4 Backend for Each Hamiltonian

Each trial re-initialises the ansatz from scratch to avoid state carry-over.  
**12 instances × 4 backends × 5 trials** = 240 total runs.

In [ ]:
NUM_TRIALS = 5
ITERS = 3
OPT_RUNS = 200
NUM_QUBITS = 9  # all instances are 9-qubit
# backend_order = ["ideal", "gate", "readout", "combined"]
backend_order = ["ideal"]

# results[instance_name] = dict of energy/time arrays
results = {}

BETA1 = 0.9
BETA2 = 0.99

for inst_name, H in hamiltonians.items():
    print(f"\n{'='*65}")
    print(f"  {inst_name}")
    print(f"{'='*65}")

    for trial in range(1, NUM_TRIALS + 1):
        print(f'  Trial {trial}/{NUM_TRIALS}', end=' ... ', flush=True)

        current_tasks = []
        for backend_name in backend_order:
            circuit_m, ansatz_m, params_m, inds_m = make_initial_ansatz(NUM_QUBITS)
            backend_config = backend_configs[backend_name]

            current_tasks.append({
                "name": backend_name,
                "func": momentum_sa_merged_noisy,
                "args": (params_m, inds_m, ansatz_m, circuit_m, H, backend_config,
                         BETA1, BETA2, ITERS, OPT_RUNS)
            })

        batch_results = execute_concurrent_benchmarks(current_tasks)   

        for backend_name in backend_order:
            e_merged = batch_results[backend_name]["output"]
            runtime = batch_results[backend_name]["runtime"]

            print(f"\t-> {backend_name: < 10} ({runtime: .1f}s) | Energy: {e_merged: .4f}")

print("\nAll trials complete")


  A: pairs + triples (u=6)
  Trial 1/5 ... 

Exception in thread Exception in thread Thread-4 (thread_worker):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1045, in _bootstrap_inner
Exception in thread Thread-5 (thread_worker):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1045, in _bootstrap_inner
Thread-3 (thread_worker):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 982, in run
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py

KeyError: 'ideal'